## Задача

Обучить RNN на каком-то текстовом датасете и генерировать новый текст с теми же паттернами, чт ои исходный.

## Реализация

### Подготовка и загрузка данных

In [239]:
#!pip3 install torch torchvision --index-url https://download.pytorch.org/whl/cu130

In [240]:
import torch

In [241]:
if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print(device)

cuda


In [242]:
# Сохраняем URL
gist_url = "https://gist.github.com/bdcb66640cc070450817686f6c818897.git"

In [243]:
# Клонируем
!git clone {gist_url}

fatal: destination path 'bdcb66640cc070450817686f6c818897' already exists and is not an empty directory.


In [244]:
with open('bdcb66640cc070450817686f6c818897//war_and_peace.ru.txt', 'r', encoding='utf-8') as file:
    dataset = file.read()

In [245]:
# Гиперпараметры
BATCH_SIZE = 64
SEQ_LENGTH = 100 # длина входной последовательности каждого примера
STRIDE = 1 # шаг, с которым будем брать последовательности
EMBEDDING_SIZE = 128
HIDDEN_SIZE = 256
NUM_LAYERS = 2
DROPOUT = 0.2
LEARNING_RATE = 0.001
N_EPOCHS = 10

## Словарь и Датасет

In [246]:
chars = sorted(set(dataset))
vocab_size = len(chars)

print(f"Всего уникальных символов: {vocab_size}")

Всего уникальных символов: 146


In [247]:
char2idx = {ch: i for i, ch in enumerate(chars)}

In [248]:
idx2char = {i: ch for i,ch in enumerate(chars)}

In [249]:
# Преобразуем весь текст в индексы
dataset_index = [char2idx[ch] for ch in dataset]

In [250]:
len_dataset_index = len(dataset_index)

In [251]:
class CharDataset(torch.utils.data.Dataset):
    def __init__(self, data,seq_len):
        self.data = data  # data - список индексов
        self.seq_len = seq_len # seq_length - длина последовательности

    # Общее количество примеров
    # __len__(): вызывается когда вы пишете len(dataset)
    def __len__(self):
        return len(self.data) -self.seq_len

    # __getitem__(index): вызывается когда вы пишете dataset[index]
    def __getitem__(self, index):
         # Берём последовательность начиная с позиции index длиной seq_len
         x = self.data[index:(index + self.seq_len)]

         # Y: та же последовательность, но со сдвигом на 1
         y = self.data[index + 1:index + self.seq_len + 1]

         x_tensor = torch.tensor(x)
         y_tensor = torch.tensor(y)
         return x_tensor,y_tensor


Train и test

In [252]:
split_idx = int(len(dataset_index) * 0.8)
train_data = dataset_index[:split_idx]
test_data = dataset_index[split_idx:]

train_dataset = CharDataset(train_data, SEQ_LENGTH)
test_dataset = CharDataset(test_data, SEQ_LENGTH)

In [253]:
BATCH_SIZE = 64

In [254]:
train_loader = torch.utils.data.DataLoader(train_dataset,batch_size=BATCH_SIZE,shuffle=True)
test_loader = torch.utils.data.DataLoader(test_dataset,batch_size=BATCH_SIZE,shuffle=False)

## Модель и обучение

In [255]:
class CharRNN(torch.nn.Module):
    def __init__(self,vocab_size_, embedding_size,hidden_size,num_layers):
        # embedding_size  Размерность эмбеддингов
        # hidden_size Количество нейронов в скрытом слое RNN
        # num_layers Количество рекуррентных слоев
        super().__init__()
        self.vocab_size = vocab_size_
        self.hidden_size = hidden_size
        self.num_layers = num_layers

        # nn.Sequential может ТОЛЬКО - Один вход -> один выход
        # Первый слой
        self.embedding = torch.nn.Embedding(
            num_embeddings=vocab_size_,
            embedding_dim=embedding_size
        )

        # Второй слой
        self.rnn = torch.nn.LSTM(
            input_size=embedding_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            dropout = 0.2 if num_layers > 1 else 0  # dropout только между слоями
        )

         # Третий слой - полносвязный
        # Преобразует скрытое состояние в логиты для каждого символа
        self.lin = torch.nn.Linear(
            in_features= hidden_size,
            out_features = vocab_size_
        )

        self.dropout = torch.nn.Dropout(0.2)

    def forward(self,x, hidden = None):
        # x  входной тензор формы (batch_size, seq_length)

        batch_size = x.size(0)
        seq_len = x.size(1)

        embedded = self.embedding(x)

        # dropout к эмбеддингам
        embedded = self.dropout(embedded)

        #Выход:
        #   - output: (batch_size, seq_length, hidden_size) - выходы на каждом шаге
        #   - hidden: (hidden_state, cell_state) для LSTM
        output, hidden = self.rnn(embedded, hidden)

        # преобразования для полносвязного слоя
        # Нужно изменить форму: из (batch_size, seq_length, hidden_size)
        # в (batch_size * seq_length, hidden_size)
        # 1. .contiguous() - Убеждаемся, что данные в памяти расположены последовательно
        # Это нужно для безопасности перед изменением формы
        # -1 - автоматический подсчёт(было 10x20x5 = 1000, стало ?x5=1000)
        output = output.contiguous().view(-1, self.hidden_size)

        # dropout к выходу RNN
        output = self.dropout(output)

        logits = self.lin(output)

        return logits,hidden


In [256]:
def train_epoch(model,dataloader,criterion,optimizer,device):
    model.train()
    clip_norm = 0.5
    total_loss =0
    total_samples =0

    for batch_idx, (inputs,targets) in enumerate(dataloader):

        inputs = inputs.to(device)
        targets = targets.to(device)

        optimizer.zero_grad()

        # НЕ передаем hidden - каждый батч обрабатывается независимо
        logits, _ = model(inputs)

        # Важно: targets должен быть 1D тензором длиной batch_size * seq_len
        loss = criterion(logits,targets.view(-1))

        loss.backward()

        # Gradient clipping (очень важно для RNN!)
        # Предотвращает "взрыв" градиентов
        torch.nn.utils.clip_grad_norm_(model.parameters(), clip_norm)

        optimizer.step()

        batch_size = inputs.size(0)
        total_loss += loss.item() * batch_size
        total_samples += batch_size

        if batch_idx % (batch_size*10*2) == 0:
            print(f'Batch {batch_idx}/{len(dataloader)} Loss: {total_loss / total_samples:.3f}')

    return total_loss / total_samples



проверка на test

In [257]:
def validate(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0
    total_samples = 0

    with torch.no_grad():
        for batch_idx, (inputs,targets) in enumerate(dataloader):
            inputs = inputs.to(device)
            targets = targets.to(device)
            logits, _ = model(inputs)
            loss = criterion(logits,targets.view(-1))

            batch_size = inputs.size(0)
            total_loss += loss.item()*batch_size
            total_samples += batch_size

    return total_loss/total_samples

In [258]:
def train(model,train_loader,test_loader,n_epochs,device):
    criterion = torch.nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(),lr=0.001)

    # Сохранение лучшей модели
    best_val_loss = None
    best_model = None

    print("Начинаем обучение...")
    for epoch in range(n_epochs):
        print(f'Epoch: {epoch+1}/{n_epochs}')
        train_loss = train_epoch(model,train_loader,criterion,optimizer,device)

        test_loss = validate(model,test_loader,criterion,device)

        if best_model is None:
            best_model=model.state_dict().copy()
            best_val_loss = test_loss

        if  test_loss < best_val_loss:
            best_model=model.state_dict().copy()
            best_val_loss = test_loss

        print(f"Train Loss: {train_loss:.4f} | Test Loss: {test_loss:.4f}")

        # Загружаем лучшую модель
        if best_model is not None:
            model.load_state_dict(best_model)
    return model

## Запуск модели

In [259]:
model = CharRNN(
    vocab_size_=vocab_size,
    embedding_size=EMBEDDING_SIZE,
    hidden_size=HIDDEN_SIZE,
    num_layers=NUM_LAYERS,
).to(device)

In [260]:
train(
    model=model,
    train_loader=train_loader,
    test_loader=test_loader,
    n_epochs=N_EPOCHS,
    device=device
)

Начинаем обучение...
Epoch: 1/20
Batch 0/18334 Loss: 4.995
Batch 1280/18334 Loss: 2.693
Batch 2560/18334 Loss: 2.641
Batch 3840/18334 Loss: 2.621
Batch 5120/18334 Loss: 2.611
Batch 6400/18334 Loss: 2.603
Batch 7680/18334 Loss: 2.598
Batch 8960/18334 Loss: 2.595
Batch 10240/18334 Loss: 2.592
Batch 11520/18334 Loss: 2.590
Batch 12800/18334 Loss: 2.588
Batch 14080/18334 Loss: 2.586
Batch 15360/18334 Loss: 2.585
Batch 16640/18334 Loss: 2.584
Batch 17920/18334 Loss: 2.583


KeyboardInterrupt: 